# PV Viewer — Workshop Starter Notebook

This notebook fetches archived LCLS process-variable (PV) data and displays it as an
interactive widget.  It is **self-contained**: no local helper modules are needed.

## Your assignment
Use agentic AI coding tools to convert this notebook into a standalone **PyDM / PyQt
application** that:
1. Shows the same plots in a proper GUI window.
2. Lets the user pick a PV and a time range with real GUI controls (combo-box, spin-box, etc.).
3. Has a *Refresh* button that re-fetches and redraws the data.

> **Tip:** start by asking the agent to read this notebook and explain what each piece does,
> then ask it to scaffold a PyDM `Display` (or a plain PyQt `QWidget`) around the same logic.

In [1]:
# ── Imports ───────────────────────────────────────────────────────────────────
import datetime as dt
from zoneinfo import ZoneInfo

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import requests
import ipywidgets as widgets
from IPython.display import display

%matplotlib widget

RuntimeError: 'widget' is not a recognised GUI loop or backend name

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────

ARCHIVER_URL = "http://lcls-archapp.slac.stanford.edu/retrieval/data/getData.json"
TIMEOUT_SECONDS = 20.0
LOCAL_TZ = ZoneInfo("America/Los_Angeles")

# PVs available in the viewer.
# Each entry: display label -> (pv_name, y-axis label)
PV_DEFS = {
    "GDET 241 — Pulse Energy (HXR)": ("GDET:FEE1:241:ENRC",                "Energy (mJ)"),
    "GMD — Pulse Energy (SXR)":      ("EM1K0:GMD:HPS:milliJoulesPerPulse", "Energy (mJ)"),
    "QUAD IN20:121 — Magnet":        ("QUAD:IN20:121:BCTRL",                "Field (kG)"),
    "BPM IN20:221 — X Position":     ("BPMS:IN20:221:X",                   "Position (mm)"),
}


In [ ]:
# ── Archive fetch ─────────────────────────────────────────────────────────────

def _to_utc_str(t: dt.datetime) -> str:
    """Convert a datetime to the ISO-8601 UTC string expected by the archiver."""
    if t.tzinfo is None:
        t = t.replace(tzinfo=LOCAL_TZ)
    return t.astimezone(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.000Z")


def fetch_pv(pv_name: str, hours_back: float = 1.0):
    """
    Fetch archived data for *pv_name* covering the last *hours_back* hours.

    Returns
    -------
    timestamps : list[datetime]
    values     : np.ndarray (float)
    """
    now = dt.datetime.now(LOCAL_TZ)
    start = now - dt.timedelta(hours=hours_back)

    response = requests.get(
        ARCHIVER_URL,
        params={"pv": pv_name, "from": _to_utc_str(start), "to": _to_utc_str(now)},
        timeout=TIMEOUT_SECONDS,
    )
    response.raise_for_status()

    payload = response.json()
    if not payload or "data" not in payload[0]:
        raise RuntimeError(f"No data returned for PV: {pv_name}")

    data = payload[0]["data"]
    secs = np.array([d["secs"] + d.get("nanos", 0) * 1e-9 for d in data])
    vals = np.array([d["val"] for d in data], dtype=float)

    # Convert UNIX epoch → aware datetime objects for matplotlib
    timestamps = [
        dt.datetime.fromtimestamp(s, tz=LOCAL_TZ) for s in secs
    ]
    return timestamps, vals

In [ ]:
# ── Plot helper ───────────────────────────────────────────────────────────────

def plot_pv(label: str, hours_back: float):
    """
    Fetch and plot the PV identified by *label* (a key in PV_DEFS).
    Returns a matplotlib Figure.
    """
    pv_name, y_label = PV_DEFS[label]

    fig, ax = plt.subplots(figsize=(10, 3))
    ax.set_title(label, fontsize=11)
    ax.set_ylabel(y_label)
    ax.set_xlabel("Time (local)")

    try:
        timestamps, values = fetch_pv(pv_name, hours_back=hours_back)
        ax.plot(timestamps, values, linewidth=0.8, color="steelblue")
        ax.xaxis.set_major_formatter(
            mdates.DateFormatter("%H:%M", tz=LOCAL_TZ)
        )
        fig.autofmt_xdate(rotation=30)
    except Exception as exc:
        ax.text(0.5, 0.5, f"Error fetching data:\n{exc}",
                ha="center", va="center", transform=ax.transAxes,
                color="firebrick", fontsize=9)

    fig.tight_layout()
    return fig

In [ ]:
# ── Interactive widget ────────────────────────────────────────────────────────

pv_dropdown = widgets.Dropdown(
    options=list(PV_DEFS.keys()),
    description="PV:",
    layout=widgets.Layout(width="420px"),
)

hours_slider = widgets.IntSlider(
    value=1,
    min=1,
    max=48,
    step=1,
    description="Hours back:",
    continuous_update=False,
    layout=widgets.Layout(width="420px"),
)

refresh_button = widgets.Button(
    description="Refresh",
    button_style="primary",
    icon="refresh",
)

output = widgets.Output()


def _update(_=None):
    with output:
        output.clear_output(wait=True)
        fig = plot_pv(pv_dropdown.value, hours_back=hours_slider.value)
        plt.show(fig)
        plt.close(fig)


pv_dropdown.observe(_update, names="value")
hours_slider.observe(_update, names="value")
refresh_button.on_click(_update)

controls = widgets.VBox([
    widgets.HBox([pv_dropdown, hours_slider, refresh_button]),
    output,
])

display(controls)
_update()  # draw immediately on first run